In [0]:
%sql

USE CATALOG `retail-sales-proj-t9`;
USE SCHEMA gold;

In [0]:
%sql

USE CATALOG `retail-sales-proj-t9`;

USE SCHEMA gold;

CREATE OR REPLACE TABLE customer_anomalies
USING DELTA
AS

SELECT

    customer_id,
    customer_name,
    email,
    city,

    CASE

        WHEN email IN (

            SELECT email

            FROM dim_customer

            GROUP BY email

            HAVING COUNT(*) > 1

        )

        THEN 'DUPLICATE_EMAIL_ANOMALY'

        ELSE 'NORMAL'

    END AS anomaly_status,

    CURRENT_TIMESTAMP() AS anomaly_detected_time

FROM dim_customer;

In [0]:
%sql

CREATE OR REPLACE TABLE customer_anomaly_alerts
USING DELTA
AS

SELECT

    customer_id,
    customer_name,
    email,

    'CUSTOMER_DUPLICATE_ALERT' AS alert_type,

    CURRENT_TIMESTAMP() AS alert_generated_time

FROM customer_anomalies

WHERE anomaly_status = 'DUPLICATE_EMAIL_ANOMALY';

In [0]:
%sql

CREATE OR REPLACE VIEW valid_customers AS

SELECT *

FROM customer_anomalies

WHERE anomaly_status = 'NORMAL';

In [0]:
%sql

CREATE OR REPLACE TABLE product_anomalies
USING DELTA
AS

WITH product_stats AS (

    SELECT

        AVG(unit_price) AS avg_price,
        STDDEV(unit_price) AS std_price

    FROM dim_product

)

SELECT

    p.product_id,
    p.product_name,
    p.category,
    p.unit_price,

    CASE

        WHEN p.unit_price >
             (ps.avg_price + 2 * ps.std_price)

        THEN 'PRICE_ANOMALY'

        ELSE 'NORMAL'

    END AS anomaly_status,

    CURRENT_TIMESTAMP() AS anomaly_detected_time

FROM dim_product p
CROSS JOIN product_stats ps;

In [0]:
%sql

CREATE OR REPLACE TABLE product_anomaly_alerts
USING DELTA
AS

SELECT

    product_id,
    product_name,
    unit_price,

    'PRODUCT_PRICE_ALERT' AS alert_type,

    CURRENT_TIMESTAMP() AS alert_generated_time

FROM product_anomalies

WHERE anomaly_status = 'PRICE_ANOMALY';

In [0]:
%sql

CREATE OR REPLACE VIEW valid_products AS

SELECT *

FROM product_anomalies

WHERE anomaly_status = 'NORMAL';

In [0]:
%sql

CREATE OR REPLACE TABLE gold.sales_anomalies
USING DELTA
AS

WITH sales_stats AS (

    SELECT
        AVG(quantity) AS avg_qty,
        STDDEV(quantity) AS std_qty

    FROM gold.fact_sales

)

SELECT

    s.transaction_id,
    s.customer_id,
    s.product_id,
    s.store_id,
    s.quantity,
    s.transaction_date,

    st.avg_qty,
    st.std_qty,

    CASE

        WHEN s.quantity >
             (st.avg_qty + 2 * st.std_qty)

        THEN 'ANOMALY'

        ELSE 'NORMAL'

    END AS anomaly_status,

    CURRENT_TIMESTAMP() AS anomaly_detected_time

FROM gold.fact_sales s
CROSS JOIN sales_stats st;

In [0]:
%sql

CREATE OR REPLACE TABLE gold.anomaly_alerts
USING DELTA
AS

SELECT

    transaction_id,
    customer_id,
    product_id,
    quantity,

    'HIGH_QUANTITY_ALERT' AS alert_type,

    CURRENT_TIMESTAMP() AS alert_generated_time

FROM gold.sales_anomalies

WHERE anomaly_status = 'ANOMALY';

In [0]:
%sql

SELECT *
FROM gold.sales_anomalies
WHERE anomaly_status = 'ANOMALY';

In [0]:
%sql

CREATE OR REPLACE VIEW gold.valid_sales AS

SELECT *
FROM gold.sales_anomalies

WHERE anomaly_status = 'NORMAL';